In [2]:
import radiation
import constants
from pytz import timezone
from datetime import datetime
from pint import Quantity
import numpy as np
from scipy import interpolate

In [2]:
nu_min = Quantity(50, "cm^(-1)").to_base_units().m
nu_max = Quantity(50000, "cm^(-1)").to_base_units().m
nu = np.arange(nu_min, nu_max, 1000)

In [ ]:
nu

In [ ]:
len(nu)

In [ ]:
%%timeit
radiation.spectral_exitance(np.array([5773], ndmin=0), nu[:])

In [5]:
ToA_incoming_exitance = radiation.spectral_exitance(np.array([5773], ndmin=0), nu[:])
len(ToA_incoming_exitance)

4995

In [ ]:
%%timeit
M_in = radiation.ground_incoming_shortwave(((constants.r_sun**2 * radiation.spectral_exitance(np.array([5773], ndmin=0), nu[:])) / constants.D_sun**2), datetime(2025, 10, 25, 12).replace(tzinfo = timezone("UTC")).astimezone(timezone("America/Chicago")), 33.16, 0)
M_in

In [6]:
current_model_time = datetime(2025, 10, 25, 12).replace(tzinfo = timezone("UTC")).astimezone(timezone("America/Chicago"))
lat = 33.16
shortwave_reduction = 0

In [ ]:
%%timeit
local_hour = current_model_time.hour + current_model_time.minute / 60 + current_model_time.second / 3600

In [13]:
local_hour = current_model_time.hour + current_model_time.minute / 60 + current_model_time.second / 3600

In [ ]:
%%timeit
solar_declination = -23.44 * np.cos((np.deg2rad(360)/365) * (int(current_model_time.strftime("%j")) + local_hour/24) + 10)

In [15]:
solar_declination = -23.44 * np.cos((np.deg2rad(360)/365) * (int(current_model_time.strftime("%j")) + local_hour/24) + 10)

In [ ]:
%%timeit
solar_hour_angle = (local_hour - 12) * (360/24)

In [17]:
solar_hour_angle = (local_hour - 12) * (360/24)

In [ ]:
%%timeit
solar_zenith_angle = np.arccos(np.sin(np.deg2rad(lat)) * np.sin(np.deg2rad(solar_declination)) + np.cos(np.deg2rad(lat)) * np.cos(np.deg2rad(solar_declination)) * np.cos(np.deg2rad(solar_hour_angle)))

In [19]:
solar_zenith_angle = np.arccos(np.sin(np.deg2rad(lat)) * np.sin(np.deg2rad(solar_declination)) + np.cos(np.deg2rad(lat)) * np.cos(np.deg2rad(solar_declination)) * np.cos(np.deg2rad(solar_hour_angle)))

In [ ]:
%%timeit
if -90 <= np.rad2deg(solar_zenith_angle) <= 90:
        incoming_shortwave = ToA_incoming_exitance * (0.6 * np.cos(solar_zenith_angle) + 0.2 * np.cos(solar_zenith_angle)**2)  * (1 - shortwave_reduction)

In [3]:
%%timeit
current_model_time = datetime(2025, 10, 25, 12).replace(tzinfo = timezone("UTC")).astimezone(timezone("America/Chicago"))
lat = 33.16
shortwave_reduction = 0
ToA_incoming_exitance = radiation.spectral_exitance(np.array([5773], ndmin=0), nu[:])
M_in = radiation.ground_incoming_shortwave(ToA_incoming_exitance, current_model_time, lat, shortwave_reduction)

82.9 μs ± 1.44 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
M_in = radiation.ground_incoming_shortwave(ToA_incoming_exitance, current_model_time, lat, shortwave_reduction)

In [8]:
%%timeit
radiation.irradiance(M_in)

51.9 μs ± 557 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [ ]:
theta_z_s = interpolate.interp1d(z[:].data, theta[current_time_index-1,:].data)(z_s[current_time_index-1].data)
u_z_s = interpolate.interp1d(z[:].data, u_bar[current_time_index-1,:].data)(z_s[current_time_index-1].data)
v_z_s = interpolate.interp1d(z[:].data, v_bar[current_time_index-1,:].data)(z_s[current_time_index-1].data)
r_v_s = interpolate.interp1d(z[:].data, r_v[current_time_index-1,:].data)(z_s[current_time_index-1].data)